# AMD AI Notebook Manual Checkpoint

Run this notebook from the AMD AI Notebook environment connected to the repo. It performs safe local checks, prints commit/manual-test instructions, then intentionally stops before live Fireworks calls, Docker push, or submission steps. After the stop, resume only when the repo is committed and you are ready to run the guarded live smoke test.

Do not paste API keys into notebook output. Use environment variables for live runs.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "Dockerfile").exists() and (candidate / "app").is_dir() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find juggernaut-router repo root from this notebook location.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.executable}")

In [ ]:
def run(cmd, *, env=None, check=True):
    print("\n$ " + " ".join(str(part) for part in cmd))
    completed = subprocess.run(
        [str(part) for part in cmd],
        cwd=REPO_ROOT,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {cmd}")
    return completed


print("Helper ready.")

## Safe Local Tests

These tests do not call Fireworks and do not require secrets. They are safe to run before committing.

In [ ]:
run([sys.executable, "scripts/run_local_quality_gate.py"])

In [ ]:
local_env = os.environ.copy()
local_env.update({
    "INPUT_PATH": "local_test/input/tasks.json",
    "OUTPUT_PATH": "local_test/output/results.json",
})

run([sys.executable, "-m", "app.main"], env=local_env)
run([sys.executable, "scripts/validate_submission_io.py", "local_test/output/results.json"])

In [ ]:
run([sys.executable, "eval/model_matrix.py", "--prompt-policies", "all"])

reports = sorted((REPO_ROOT / "eval_runs").glob("model_matrix_*.md"))
if reports:
    print(f"Latest model matrix report: {reports[-1]}")

In [ ]:
run([sys.executable, "eval/router_config_sweep.py", "--accuracy-threshold", "0.80"])

router_reports = sorted((REPO_ROOT / "eval_runs").glob("router_sweep_*.md"))
if router_reports:
    print(f"Latest router sweep report: {router_reports[-1]}")

## Commit And Manual-Test Checkpoint

The next cell intentionally stops the notebook. Before continuing:

1. Review the changed files.
2. Commit the repo.
3. Manually test the app from the AMD AI Notebook terminal.
4. Resume from the cells after the stop only when you are ready for optional live checks.

In [ ]:
run(["git", "status", "--short"], check=False)

print("Suggested commit commands:")
print("git add README.md docs eval app scripts tests notebooks .gitignore .dockerignore Dockerfile requirements.txt")
print('git commit -m "Run AMD notebook local checkpoint"')

raise SystemExit(
    "MANUAL CHECKPOINT: stop here. Commit and manually test in the AMD AI Notebook terminal, then resume only when ready."
)

## Resume After Manual Commit

Run these cells only after committing and manually testing. They are guarded so live/expensive work does not run by accident.

In [ ]:
run([sys.executable, "scripts/check_live_eval_env.py", "--print-models"])

In [ ]:
RUN_LIVE_FIREWORKS_SMOKE = False

if not RUN_LIVE_FIREWORKS_SMOKE:
    raise SystemExit("Live Fireworks smoke is disabled. Set RUN_LIVE_FIREWORKS_SMOKE = True only when ready to spend credits/tokens.")

first_model = "minimax-m3" if "minimax-m3" in os.environ["ALLOWED_MODELS"] else os.environ["ALLOWED_MODELS"].split(",")[0].strip()
run([
    sys.executable,
    "eval/model_matrix.py",
    "--live",
    "--limit",
    "2",
    "--models",
    first_model,
    "--prompt-policies",
    "original",
    "--max-tokens",
    "128",
])

In [ ]:
RUN_DOCKER_SMOKE = False

if not RUN_DOCKER_SMOKE:
    raise SystemExit("Docker smoke is disabled. Set RUN_DOCKER_SMOKE = True only when the notebook environment has Docker available.")

run([sys.executable, "scripts/check_docker_runtime.py"])